# DSC288R -- Comprehensive Exploratory Data Analysis
## Multi-Agent Graph RAG System for Explainable Financial Decision Support

**Group 10:** Harsh Arya, Gabrielle Despaigne, Camila Paik, Raghav Vasappanavara  
**Analysis Date:** February 2026

---

### Stock Universe

60 tickers sourced from **Finnhub** across three high-activity sectors:

| Sector | Tickers | Examples |
|--------|---------|----------|
| Finance | 20 | JPM, GS, V, MA, BAC |
| Semiconductor | 20 | NVDA, AMD, TSM, AVGO, QCOM |
| Biotech | 20 | AMGN, GILD, REGN, VRTX, MRNA |

Historical OHLCV prices are loaded from **FNSPID** for the 58 tickers with available price history.  
Fundamentals, earnings, and news come from **Finnhub**.  
Market context from **S&P 500** (Yahoo Finance).

### TA Comments Addressed

| TA # | Comment | Section |
|------|---------|--------|
| 1.2 | Outlier cutoff justification | 4 |
| 1.3 | Buy/hold/sell threshold definition | 5 |
| 2.1 | Concrete anomaly table with counts | 11 |
| 2.2 | Correlation meaningfulness | 9 |
| 3.1 | Sentiment feature pipeline | 7 |
| 4.1 | Reproducible stock selection | 2 |
| 4.4 | FinQA connection to explanations | 8 |

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'
FH   = RAW / 'finnhub_stocks'
OUT  = Path('outputs')
OUT.mkdir(exist_ok=True)

sector_index = json.loads((FH / '_sector_index.json').read_text())
TICKERS = list(sector_index.keys())
SECTORS = sorted(set(sector_index.values()))
print(f'Finnhub universe: {len(TICKERS)} tickers across {SECTORS}')

## 2. Load & Align Price History (TA 4.1)

> *TA: "The report does not state the exact subset or how stocks were chosen."*

**Selection rule:** 60 tickers chosen for sector diversity and market significance across finance, semiconductor, and biotech. Prices loaded from FNSPID for the 58 tickers with available history; 2 tickers (ARM, MRNA) lack FNSPID coverage due to recent IPOs.

In [ ]:
price_base = RAW / 'fnspid' / 'Stock_price' / 'full_history'

frames = []
matched, missing = [], []
for ticker in TICKERS:
    hits = [f for f in price_base.rglob(f'{ticker}.csv') if not f.name.startswith('._')]
    if hits:
        try:
            df = pd.read_csv(hits[0], parse_dates=['date'])
            df['ticker'] = ticker
            df['sector'] = sector_index[ticker]
            frames.append(df)
            matched.append(ticker)
        except Exception as e:
            missing.append((ticker, str(e)))
    else:
        missing.append((ticker, 'no CSV'))

prices = pd.concat(frames, ignore_index=True).sort_values(['ticker', 'date'])
print(f'Loaded {len(matched)} / {len(TICKERS)} tickers  |  {len(prices):,} rows')
print(f'Date range: {prices["date"].min().date()} -- {prices["date"].max().date()}')
if missing:
    print(f'Missing: {[m[0] for m in missing]}')

print(f'\nRows per sector:')
print(prices.groupby('sector').agg(tickers=('ticker','nunique'), rows=('close','size')).to_string())

## 3. Data Quality

In [ ]:
quality = pd.DataFrame({
    'dtype': prices.dtypes,
    'non_null': prices.count(),
    'null_pct': (prices.isnull().sum() / len(prices) * 100).round(2),
})
print(quality.to_string())

rows_per_ticker = prices.groupby('ticker').size()
print(f'\nTrading days per ticker: min={rows_per_ticker.min()}, '
      f'median={rows_per_ticker.median():.0f}, max={rows_per_ticker.max()}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Price Distributions (Finnhub Universe)', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flat[:5], ['open', 'high', 'low', 'close', 'volume']):
    if col in prices.columns:
        data = prices[col].dropna()
        ax.hist(data.clip(upper=data.quantile(0.99)), bins=80, edgecolor='none', alpha=0.7)
        ax.set_title(col.title())
        med = data.median()
        ax.axvline(med, color='red', ls='--', lw=1.2, label=f'Median={med:,.1f}')
        ax.legend(fontsize=8)

for sector in SECTORS:
    sub = rows_per_ticker[prices.groupby('ticker')['sector'].first() == sector]
    axes[1, 2].hist(sub, bins=20, alpha=0.5, label=sector)
axes[1, 2].legend(fontsize=8)
axes[1, 2].set_title('Trading Days per Ticker')
axes[1, 2].set_xlabel('Days')

plt.tight_layout()
plt.savefig(OUT / '01_price_distributions.png', bbox_inches='tight')
plt.show()

## 4. Outlier Analysis (TA 1.2)

> *TA: "Why 50%? How many are valid events vs errors?"*

In [ ]:
prices['daily_return'] = prices.groupby('ticker')['close'].pct_change() * 100
returns = prices['daily_return'].dropna()

print(f'Records with returns: {len(returns):,}')
print(f'Range: {returns.min():.1f}% to {returns.max():.1f}%')

print(f'\n{"Threshold":>10} {"Count":>8} {"% Data":>8}')
for t in [10, 20, 50, 100]:
    n = (returns.abs() > t).sum()
    print(f'{t:>9}%  {n:>8,} {n/len(returns)*100:>7.3f}%')

outliers_50 = prices[prices['daily_return'].abs() > 50]
if len(outliers_50) > 0:
    print(f'\nOutlier tickers (>50%):')
    for tk, cnt in outliers_50['ticker'].value_counts().head(10).items():
        print(f'  {tk:<8} {cnt:>4} records')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Outlier Analysis (TA 1.2)', fontsize=14, fontweight='bold')

axes[0].hist(returns.clip(-30, 30), bins=200, edgecolor='none', alpha=0.7)
for v, c in [(-50, 'red'), (50, 'red'), (-2, 'orange'), (2, 'orange')]:
    axes[0].axvline(v, color=c, ls='--', lw=1.5)
axes[0].set_title('Daily Returns (clipped +/-30%)')
axes[0].set_xlabel('Return (%)')

extreme = returns[returns.abs() > 10]
if len(extreme) > 0:
    axes[1].hist(extreme.clip(-100, 100), bins=60, edgecolor='none', alpha=0.7, color='#d62728')
    axes[1].axvline(-50, color='black', ls='--', lw=2, label='-50%')
    axes[1].axvline(50, color='black', ls='--', lw=2, label='+50%')
    axes[1].legend()
axes[1].set_title('Extreme Returns (|r| > 10%)')

# Per-sector volatility
sect_vol = prices.groupby('sector')['daily_return'].std()
sect_vol.plot.bar(ax=axes[2], color=['#2ca02c','#1f77b4','#ff7f0e'])
axes[2].set_title('Daily Return Volatility by Sector')
axes[2].set_ylabel('Std Dev (%)')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(OUT / '02_outlier_analysis.png', bbox_inches='tight')
plt.show()

**Finding (TA 1.2):** With the curated Finnhub universe (large-cap, liquid stocks), extreme >50% moves are rare and likely data errors. The 50% threshold is conservative -- all real events (COVID-19, earnings) sit in the 10-50% range.

## 5. Target Variable (TA 1.3)

> *TA: "What are the exact thresholds? Fixed or tuned?"*

In [ ]:
prices['next_day_return'] = prices.groupby('ticker')['close'].pct_change().shift(-1) * 100
clean = prices.dropna(subset=['next_day_return']).copy()
clean = clean[clean['daily_return'].abs() <= 50]

clean['target'] = 'hold'
clean.loc[clean['next_day_return'] > 2, 'target'] = 'buy'
clean.loc[clean['next_day_return'] < -2, 'target'] = 'sell'

td = clean['target'].value_counts()
tp = clean['target'].value_counts(normalize=True) * 100
print('=== Target (fixed +/-2%) ===')
for l in ['buy', 'hold', 'sell']:
    if l in td.index:
        print(f'  {l.upper():<6} {td[l]:>10,}  ({tp[l]:.1f}%)')
print(f'  Total: {len(clean):,}')

print(f'\n=== Threshold Sensitivity ===')
print(f'{"Thresh":>8} {"Buy%":>7} {"Hold%":>7} {"Sell%":>7}')
for t in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 5.0]:
    b = (clean['next_day_return'] > t).mean() * 100
    s = (clean['next_day_return'] < -t).mean() * 100
    h = 100 - b - s
    mark = ' <--' if t == 2.0 else ''
    print(f'{t:>7.1f}% {b:>6.1f}% {h:>6.1f}% {s:>6.1f}%{mark}')

print(f'\n=== Target by Sector ===')
ct = pd.crosstab(clean['sector'], clean['target'], normalize='index') * 100
print(ct.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Target Variable (TA 1.3): Fixed +/-2%', fontsize=14, fontweight='bold')

axes[0].hist(clean['next_day_return'].clip(-10, 10), bins=200, edgecolor='none', alpha=0.7)
axes[0].axvline(-2, color='red', ls='--', lw=2, label='SELL < -2%')
axes[0].axvline(2, color='green', ls='--', lw=2, label='BUY > +2%')
axes[0].axvspan(-2, 2, alpha=0.08, color='gray')
axes[0].legend(fontsize=9)
axes[0].set_title('Next-Day Return')
axes[0].set_xlabel('Return (%)')

target_colors = {'buy': '#2ca02c', 'hold': '#1f77b4', 'sell': '#d62728'}
order = [l for l in ['buy', 'hold', 'sell'] if l in td.index]
td[order].plot.pie(ax=axes[1], colors=[target_colors[l] for l in order],
                   autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Balance')
axes[1].set_ylabel('')

ct.plot.bar(ax=axes[2], color=[target_colors.get(c, 'gray') for c in ct.columns])
axes[2].set_title('Target % by Sector')
axes[2].set_ylabel('%')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Target')

plt.tight_layout()
plt.savefig(OUT / '03_target_thresholds.png', bbox_inches='tight')
plt.show()

**Finding (TA 1.3):** Fixed +/-2% thresholds. Biotech shows higher buy/sell frequency (more volatile), while finance is more concentrated in HOLD -- consistent with sector risk profiles.

## 6. Finnhub Fundamentals: Profiles, Earnings & News

In [ ]:
# --- Profiles ---
profiles = []
for ticker in TICKERS:
    sector = sector_index[ticker]
    pf = FH / sector / ticker / 'profile.json'
    if pf.exists():
        try:
            p = json.loads(pf.read_text(encoding='utf-8'))
            if isinstance(p, dict) and p:
                p['sector'] = sector
                p['_ticker'] = ticker
                profiles.append(p)
        except Exception:
            pass

prof_df = pd.DataFrame(profiles)
prof_df['mktcap_B'] = pd.to_numeric(prof_df.get('marketCapitalization', 0), errors='coerce') / 1000
print(f'Profiles: {len(prof_df)}')
print('\nMarket Cap ($B) by Sector:')
print(prof_df.groupby('sector')['mktcap_B'].agg(['count','mean','median','max']).round(1).to_string())

# --- Earnings ---
earn_rows = []
for ticker in TICKERS:
    sector = sector_index[ticker]
    ef = FH / sector / ticker / 'earnings.json'
    if ef.exists():
        try:
            data = json.loads(ef.read_text(encoding='utf-8'))
            if isinstance(data, list):
                for e in data:
                    e['ticker'] = ticker
                    e['sector'] = sector
                    earn_rows.append(e)
        except Exception:
            pass

earn_df = pd.DataFrame(earn_rows)
earn_df['surprise'] = pd.to_numeric(earn_df.get('surprise', 0), errors='coerce')
earn_df['surprisePercent'] = pd.to_numeric(earn_df.get('surprisePercent', 0), errors='coerce')
print(f'\nEarnings: {len(earn_df):,} records, {earn_df["ticker"].nunique()} tickers')
print('\nEarnings Surprise % by Sector:')
print(earn_df.groupby('sector')['surprisePercent'].agg(['mean','median','std']).round(2).to_string())

In [ ]:
# --- News ---
news_rows = []
for ticker in TICKERS:
    sector = sector_index[ticker]
    nf = FH / sector / ticker / 'news.json'
    if nf.exists():
        try:
            arts = json.loads(nf.read_text(encoding='utf-8'))
            n = len(arts) if isinstance(arts, list) else 0
        except Exception:
            n = 0
        news_rows.append({'ticker': ticker, 'sector': sector, 'articles': n})

news_df = pd.DataFrame(news_rows)
print(f'News: {news_df["articles"].sum():,} articles across {len(news_df)} tickers')
print('\nNews Volume by Sector (30-day window):')
print(news_df.groupby('sector')['articles'].agg(['count','mean','median','sum']).round(0).to_string())

# --- News Sentiment ---
sent_rows = []
for ticker in TICKERS:
    sector = sector_index[ticker]
    sf = FH / sector / ticker / 'news_sentiment.json'
    if sf.exists():
        try:
            d = json.loads(sf.read_text(encoding='utf-8'))
            if isinstance(d, dict) and d:
                s = d.get('sentiment', {})
                b = d.get('buzz', {})
                sent_rows.append({
                    'ticker': ticker, 'sector': sector,
                    'bullish': s.get('bullishPercent', 0),
                    'bearish': s.get('bearishPercent', 0),
                    'buzz': b.get('buzz', 0),
                    'articles_week': b.get('articlesInLastWeek', 0),
                })
        except Exception:
            pass

sent_df = pd.DataFrame(sent_rows)
if len(sent_df) > 0:
    sent_df['net_sentiment'] = sent_df['bullish'] - sent_df['bearish']
    print(f'\nSentiment: {len(sent_df)} tickers')
    print(sent_df.groupby('sector')[['bullish','bearish','net_sentiment','buzz']].mean().round(3).to_string())

In [ ]:
sector_colors = {'biotech': '#2ca02c', 'finance': '#1f77b4', 'semiconductor': '#ff7f0e'}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Finnhub Fundamentals (60 Tickers, 3 Sectors)', fontsize=14, fontweight='bold')

# Market cap
for s in SECTORS:
    sub = prof_df[prof_df['sector'] == s]['mktcap_B'].dropna()
    if len(sub) > 0:
        axes[0,0].hist(sub, bins=15, alpha=0.5, label=s, color=sector_colors.get(s))
axes[0,0].legend(fontsize=8)
axes[0,0].set_title('Market Cap ($B)')

# Earnings surprise
for s in SECTORS:
    sub = earn_df[earn_df['sector'] == s]['surprisePercent'].dropna()
    if len(sub) > 0:
        axes[0,1].hist(sub.clip(-50, 100), bins=40, alpha=0.5, label=s, color=sector_colors.get(s))
axes[0,1].legend(fontsize=8)
axes[0,1].set_title('Earnings Surprise %')

# News volume
top_news = news_df.nlargest(20, 'articles')
bar_c = [sector_colors.get(s, 'gray') for s in top_news['sector']]
axes[0,2].barh(top_news['ticker'], top_news['articles'], color=bar_c)
axes[0,2].set_title('Top 20 by News Volume (30d)')
axes[0,2].invert_yaxis()

# News per sector boxplot
news_df.boxplot(column='articles', by='sector', ax=axes[1,0], patch_artist=True)
axes[1,0].set_title('Articles per Ticker')
axes[1,0].set_xlabel('')
fig.suptitle('Finnhub Fundamentals (60 Tickers, 3 Sectors)', fontsize=14, fontweight='bold')

# Sentiment scatter
if len(sent_df) > 0:
    for s in SECTORS:
        sub = sent_df[sent_df['sector'] == s]
        axes[1,1].scatter(sub['bullish'], sub['bearish'], label=s,
                          color=sector_colors.get(s), s=60, alpha=0.7)
    axes[1,1].plot([0,1],[0,1],'k--',alpha=0.2)
    axes[1,1].legend(fontsize=8)
    axes[1,1].set_title('Bullish vs Bearish %')
    axes[1,1].set_xlabel('Bullish'); axes[1,1].set_ylabel('Bearish')

# Net sentiment ranked
if len(sent_df) > 0:
    ss = sent_df.sort_values('net_sentiment')
    bar_c2 = ['#d62728' if v < 0 else '#2ca02c' for v in ss['net_sentiment']]
    axes[1,2].barh(ss['ticker'], ss['net_sentiment'], color=bar_c2)
    axes[1,2].axvline(0, color='black', lw=0.5)
    axes[1,2].set_title('Net Sentiment (Bullish - Bearish)')
    axes[1,2].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig(OUT / '04_finnhub_fundamentals.png', bbox_inches='tight')
plt.show()

## 7. Financial Phrasebank -- Sentiment Baseline (TA 3.1)

> *TA: "Which sentiment model? What output? How aggregated?"*

In [ ]:
pb_dir = RAW / 'financial_phrasebank' / 'data' / 'FinancialPhraseBank-v1.0'
pb_files = sorted(pb_dir.glob('Sentences_*.txt'))

all_pb = {}
for f in pb_files:
    lines = f.read_text(encoding='latin-1').strip().split('\n')
    parsed = [{'sentence': l.rsplit('@',1)[0].strip(), 'label': l.rsplit('@',1)[1].strip()}
              for l in lines if '@' in l]
    all_pb[f.stem] = pd.DataFrame(parsed)
    print(f'  {f.stem}: {len(parsed):,}')

pb = all_pb.get('Sentences_AllAgree', list(all_pb.values())[0])
pb['words'] = pb['sentence'].str.split().str.len()
print(f'\nAllAgree: {len(pb):,} sentences')
print(pb['label'].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Financial Phrasebank (TA 3.1)', fontsize=14, fontweight='bold')

label_colors = {'positive': '#2ca02c', 'neutral': '#1f77b4', 'negative': '#d62728'}
vc = pb['label'].value_counts()
vc.plot.bar(ax=axes[0], color=[label_colors.get(l,'gray') for l in vc.index])
axes[0].set_title('Sentiment (AllAgree)')
axes[0].tick_params(axis='x', rotation=0)

for lab, col in label_colors.items():
    sub = pb[pb['label']==lab]['words']
    if len(sub)>0:
        axes[1].hist(sub, bins=40, alpha=0.5, label=lab, color=col)
axes[1].legend()
axes[1].set_title('Sentence Length by Sentiment')
axes[1].set_xlabel('Words')

agreement = {k.replace('Sentences_',''): len(v) for k,v in all_pb.items()}
pd.Series(agreement).plot.bar(ax=axes[2], color='#9467bd')
axes[2].set_title('Sentences by Agreement Level')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUT / '05_phrasebank.png', bbox_inches='tight')
plt.show()

**Finding (TA 3.1):** Sentiment pipeline: `article text -> FinBERT -> score [-1,+1] -> mean per stock-day`. Phrasebank validates the model. AllAgree (100% consensus) is 59% neutral, 28% positive, 13% negative.

## 8. FinQA (TA 4.4)

> *TA: "How does FinQA connect to explanations?"*

In [ ]:
fq_data = {}
for f in sorted((RAW / 'finqa').glob('finqa_*.json')):
    split = f.stem.replace('finqa_', '')
    recs = json.loads(f.read_text(encoding='utf-8'))
    fq_data[split] = recs
    print(f'  {split}: {len(recs):,}')
total_fq = sum(len(v) for v in fq_data.values())
print(f'  Total: {total_fq:,}')

train_fq = fq_data.get('train', [])
if train_fq:
    qs = [r['qa']['question'] for r in train_fq if 'qa' in r]
    q_lens = [len(q.split()) for q in qs]
    starters = Counter(q.split()[0].lower() for q in qs)
    ops = Counter()
    for r in train_fq:
        prog = r.get('qa',{}).get('program','')
        for op in ['subtract','divide','add','multiply','greater','table_']:
            if op in prog.lower(): ops[op] += 1

    print(f'\nQuestion starters: {dict(starters.most_common(8))}')
    print(f'Operations: {dict(ops.most_common())}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('FinQA (TA 4.4)', fontsize=14, fontweight='bold')

pd.Series({k: len(v) for k,v in fq_data.items()}).plot.bar(ax=axes[0], color='#17becf')
axes[0].set_title('Records by Split')
axes[0].tick_params(axis='x', rotation=0)

if train_fq:
    axes[1].hist(q_lens, bins=40, edgecolor='none', alpha=0.7, color='#17becf')
    axes[1].set_title('Question Length')
    axes[1].set_xlabel('Words')

    pd.Series(dict(starters.most_common(10))).plot.barh(ax=axes[2], color='#17becf')
    axes[2].set_title('Question Types')
    axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(OUT / '06_finqa.png', bbox_inches='tight')
plt.show()

**Finding (TA 4.4):** FinQA tests **grounded numerical reasoning** over financial tables -- the same capability our Graph RAG needs for explanations. We use it to evaluate reasoning faithfulness, not prediction accuracy.

## 9. S&P 500 Correlation (TA 2.2)

> *TA: "Correlation of 0.023 is very small. Is it meaningful?"*

In [ ]:
sp = pd.read_csv(RAW / 'yahoo_sp500' / 'sp500_1999_2023.csv')
sp['date'] = pd.to_datetime(sp['Date'].str[:10])
sp['sp500_return'] = sp['Close'].pct_change() * 100
sp = sp[['date','sp500_return']].dropna()
print(f'S&P 500: {len(sp):,} days')

merged = clean.merge(sp, on='date', how='inner')
print(f'Merged: {len(merged):,} records')

In [ ]:
from scipy import stats
from sklearn.metrics import mutual_info_score

x = merged['sp500_return'].values
y = merged['next_day_return'].values

r_p, p_p = stats.pearsonr(x, y)
print(f'1. Pearson r = {r_p:.4f}  (p = {p_p:.2e})')
print(f'   Wrong metric for categorical target -- deflates coefficient\n')

print('2. Point-Biserial (correct metric):')
for lab in ['buy','hold','sell']:
    if lab in merged['target'].values:
        bvec = (merged['target']==lab).astype(int)
        r_pb, p_pb = stats.pointbiserialr(bvec, x)
        sig = '***' if p_pb<1e-3 else '**' if p_pb<0.01 else '*'
        print(f'   is_{lab}: r={r_pb:+.4f}  p={p_pb:.2e} {sig}')

sp_bins = pd.qcut(merged['sp500_return'], q=20, labels=False, duplicates='drop')
mi = mutual_info_score(merged['target'], sp_bins)
print(f'\n3. Mutual Information = {mi:.4f} nats')

merged['mkt_dir'] = pd.cut(merged['sp500_return'],
                           bins=[-np.inf, -0.5, 0.5, np.inf],
                           labels=['Down','Flat','Up'])
ct = pd.crosstab(merged['mkt_dir'], merged['target'])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
cv = np.sqrt(chi2 / (len(merged) * (min(ct.shape)-1)))
print(f'\n4. Chi-squared = {chi2:.1f}  p = {p_chi:.2e}  Cramer\'s V = {cv:.4f}')
print('\n   Target % by market direction:')
print((ct.div(ct.sum(axis=1), axis=0)*100).round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('S&P 500 Correlation (TA 2.2)', fontsize=14, fontweight='bold')

samp = merged.sample(min(5000, len(merged)), random_state=42)
for lab, col in target_colors.items():
    m = samp['target']==lab
    if m.any():
        axes[0].scatter(samp.loc[m,'sp500_return'], samp.loc[m,'next_day_return'],
                        alpha=0.2, s=5, color=col, label=lab)
axes[0].legend(markerscale=5)
axes[0].set_xlim(-5,5); axes[0].set_ylim(-10,10)
axes[0].set_xlabel('S&P 500 Return (%)')
axes[0].set_ylabel('Next-Day Return (%)')
axes[0].set_title(f'Scatter (r={r_p:.4f})')

ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot.bar(ax=axes[1], color=[target_colors.get(c,'gray') for c in ct_pct.columns])
axes[1].set_title('Target % by Market Dir')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=0)

num_cols = [c for c in ['close','volume','daily_return','next_day_return','sp500_return']
            if c in merged.columns]
sns.heatmap(merged[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[2], square=True, cbar_kws={'shrink':0.8})
axes[2].set_title('Correlation Matrix')

plt.tight_layout()
plt.savefig(OUT / '07_correlation.png', bbox_inches='tight')
plt.show()

**Finding (TA 2.2):** Pearson r is misleading for categorical targets. Point-biserial, MI, and chi-squared all confirm the S&P 500 relationship is **real and significant**. Down-market days show higher next-day buy probability (mean-reversion signal).

## 10. Temporal Analysis

In [ ]:
clean['year'] = clean['date'].dt.year
yearly = clean.groupby('year').agg(
    records=('close','size'),
    avg_close=('close','mean'),
    return_std=('daily_return','std'),
    buy_pct=('target', lambda x: (x=='buy').mean()*100),
    sell_pct=('target', lambda x: (x=='sell').mean()*100),
).reset_index()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Temporal Analysis', fontsize=14, fontweight='bold')

axes[0,0].bar(yearly['year'], yearly['records'], color='#1f77b4')
axes[0,0].set_title('Records per Year')

axes[0,1].plot(yearly['year'], yearly['avg_close'], 'o-', color='#2ca02c')
axes[0,1].set_title('Avg Close')

axes[0,2].plot(yearly['year'], yearly['return_std'], 'o-', color='#d62728')
axes[0,2].set_title('Return Volatility')
axes[0,2].axhline(yearly['return_std'].mean(), ls='--', color='gray', alpha=0.5)

axes[1,0].bar(yearly['year'], yearly['buy_pct'], label='Buy', color='#2ca02c', alpha=0.7)
axes[1,0].bar(yearly['year'], yearly['sell_pct'], bottom=yearly['buy_pct'],
              label='Sell', color='#d62728', alpha=0.7)
axes[1,0].legend()
axes[1,0].set_title('Buy/Sell %')

# Sector volatility over time
for s in SECTORS:
    sub = clean[clean['sector']==s].groupby('year')['daily_return'].std()
    axes[1,1].plot(sub.index, sub.values, 'o-', label=s, color=sector_colors.get(s))
axes[1,1].legend(fontsize=8)
axes[1,1].set_title('Volatility by Sector')

# Monthly volatility heatmap
recent = clean[clean['year'] >= 2018].copy()
if len(recent) > 0:
    recent['month'] = recent['date'].dt.month
    vp = recent.groupby(['year','month'])['daily_return'].std().unstack()
    sns.heatmap(vp, cmap='YlOrRd', ax=axes[1,2], cbar_kws={'shrink':0.8})
    axes[1,2].set_title('Monthly Volatility (2018+)')

plt.tight_layout()
plt.savefig(OUT / '08_temporal.png', bbox_inches='tight')
plt.show()

## 11. Anomaly Summary (TA 2.1)

> *TA: "Add a concrete anomaly table with counts per data type."*

In [ ]:
n_total = len(prices.dropna(subset=['daily_return']))
n_out50 = int((prices['daily_return'].abs() > 50).sum())
n_ext   = int(((prices['daily_return'].abs() > 10) & (prices['daily_return'].abs() <= 50)).sum())
n_zvol  = int((prices['volume'] <= 0).sum()) if 'volume' in prices.columns else 0

prices['_vma20'] = prices.groupby('ticker')['volume'].transform(
    lambda x: x.rolling(20, min_periods=1).mean())
prices['_vratio'] = prices['volume'] / prices['_vma20'].replace(0, np.nan)
n_vspike = int((prices['_vratio'] >= 5).sum())
n_no_news = int((news_df['articles']==0).sum()) if len(news_df) > 0 else 0

rows = [
    ['Stock Prices', 'Daily change > 50%',       f'{n_out50:,} ({n_out50/max(n_total,1)*100:.2f}%)', 'Removed (likely data errors)'],
    ['Stock Prices', 'Extreme returns 10-50%',    f'{n_ext:,} ({n_ext/max(n_total,1)*100:.1f}%)',     'KEPT -- real events'],
    ['Stock Prices', 'Zero / negative volume',    f'{n_zvol:,}',                                      'Removed'],
    ['Volume',       'Volume >= 5x 20-day MA',    f'{n_vspike:,}',                                    'KEPT via volume_ratio'],
    ['Phrasebank',   'Neutral class dominance',   '~59% of AllAgree',                                 'Stratified sampling'],
    ['Finnhub News', 'Tickers with 0 articles',   f'{n_no_news}',                                     'Expected (small-cap)'],
    ['FinQA',        'Multi-step reasoning',      f'{total_fq:,} Q-A pairs',                          'Faithfulness eval'],
]

anom = pd.DataFrame(rows, columns=['Data Type','Anomaly / Rule','Count / Rate','Action'])
print('=== Anomaly Summary (TA 2.1) ===')
print(anom.to_string(index=False))
anom.to_csv(OUT / '10_anomaly_table.csv', index=False)

## 12. Critical Insights

In [ ]:
insights = {
    'stock_universe': {
        'source': 'Finnhub 60 tickers, 3 sectors',
        'price_source': 'FNSPID (58/60 matched)',
        'records': int(len(prices)),
        'sectors': SECTORS,
        'tickers_per_sector': {s: sum(1 for t in TICKERS if sector_index[t]==s) for s in SECTORS},
    },
    'target': {
        'threshold': '+/-2% fixed',
        'buy_pct': round(float(tp.get('buy',0)),1),
        'hold_pct': round(float(tp.get('hold',0)),1),
        'sell_pct': round(float(tp.get('sell',0)),1),
    },
    'outliers': {
        'threshold': '50%',
        'removed': n_out50,
        'pct': round(n_out50/max(n_total,1)*100,3),
    },
    'sp500_correlation': {
        'pearson_r': round(float(r_p),4),
        'chi2': round(float(chi2),1),
        'cramers_v': round(float(cv),4),
        'mutual_info': round(float(mi),4),
    },
    'finnhub_data': {
        'profiles': int(len(prof_df)),
        'earnings_records': int(len(earn_df)),
        'total_news_articles': int(news_df['articles'].sum()),
        'sentiment_tickers': int(len(sent_df)),
    },
    'temporal_split': {
        'train': '2009--2021', 'val': '2022', 'test': '2023',
        'method': 'fixed cutoff, no shuffling',
    },
    'evaluation': {
        'primary': 'Macro F1',
        'secondary': ['per-class P/R', 'confusion matrix', 'backtest return'],
        'explanation': ['citation correctness', 'RAGAS faithfulness', 'FinQA reasoning'],
    },
    'sentiment_pipeline': {
        'model': 'FinBERT',
        'training_data': 'Financial Phrasebank (AllAgree)',
        'output': '[-1, +1]',
        'aggregation': 'mean per stock-day',
    },
}

with open(OUT / '09_eda_insights.json', 'w') as f:
    json.dump(insights, f, indent=2, default=str)

print('=== CRITICAL INSIGHTS ===')
print()
print(f'1. STOCK UNIVERSE: {len(matched)} tickers from Finnhub across')
print(f'   {SECTORS}. Principled selection replaces arbitrary alphabetical pick.')
print(f'   {len(prices):,} total price records from FNSPID.')
print()
print(f'2. SECTOR DIFFERENCES: Biotech has highest volatility and buy/sell')
print(f'   frequency. Finance is most stable. Sector is a strong feature.')
print()
print(f'3. TARGET: Fixed +/-2%. ~{tp.get("buy",0):.0f}% buy, ~{tp.get("hold",0):.0f}% hold,')
print(f'   ~{tp.get("sell",0):.0f}% sell. Macro F1 handles imbalance.')
print()
print(f'4. OUTLIERS: {n_out50} records (>{n_out50/max(n_total,1)*100:.2f}%) removed at 50%.')
print(f'   All real events in 10-50% range preserved.')
print()
print(f'5. S&P 500: Pearson r misleading. Chi2={chi2:.0f}, MI={mi:.4f}')
print(f'   confirm real signal. Mean-reversion: down market -> more buys.')
print()
print(f'6. FINNHUB RICHNESS: {len(prof_df)} profiles, {len(earn_df):,} earnings,')
print(f'   {news_df["articles"].sum():,} news articles, {len(sent_df)} sentiment scores.')
print(f'   Enables cross-dataset features (earnings surprise + news volume).')
print()
print(f'7. SENTIMENT: FinBERT validated on Phrasebank ({len(pb):,} sentences).')
print(f'   Pipeline: text -> [-1,+1] -> mean/stock-day.')
print()
print(f'8. FINQA: {total_fq:,} Q-A pairs test reasoning faithfulness.')
print()
print(f'Saved: {OUT / "09_eda_insights.json"}')

In [ ]:
print('=== Generated Files ===')
for f in sorted(OUT.glob('*')):
    sz = f.stat().st_size
    unit = 'MB' if sz > 1024*1024 else 'KB'
    val = sz/1024/1024 if sz > 1024*1024 else sz/1024
    print(f'  {f.name:<40} {val:>6.1f} {unit}')